# IaC 第2周:Terraform 进阶

> **学习目标**:掌握 module、for_each/count、workspace、remote state 等生产级用法

---

## Day 8:Module — 可复用的基础设施组件

### Module 的核心价值

- 写一次 module,多环境调用
- 参数化差异部分
- 改动 module 一处,所有使用方受益

```
modules/nginx/
├── main.tf          # 核心资源定义
├── variables.tf     # 输入变量定义
├── outputs.tf       # 输出定义
└── README.md        # 使用说明 (推荐)
```

```hcl
module "web_nginx" {
  source      = "./modules/nginx"
  name        = "web"
  environment = var.environment
  port        = 8080
}
```

## Day 9:for_each vs count

```hcl
# for_each: key-based,稳定的(推荐)
resource "docker_container" "app" {
  for_each = var.instances    # map: { "web-1" = {...}, "web-2" = {...} }
  name     = each.key
  image    = each.value.image
}

# count: index-based,会位移(不推荐当 key 重要时)
resource "docker_container" "app" {
  count = var.instance_count
  name  = "app-${count.index}"
}
```

for_each 优势:key 确定,增删某个元素不影响其他。
count 问题:从中间删除一个,后面的 index 全部位移 = 销毁+重建

## Day 10:条件表达式与函数

```hcl
count  = var.env == "prod" ? 3 : 1                                  # 三元表达式
region = coalesce(var.region, "us-east-1")                          # 取第一个非空值
size   = try(var.custom_size, var.default_size)                     # 安全取值
tags   = merge(var.default_tags, var.custom_tags)                   # 合并 map
```

## Day 11:for 表达式

```hcl
# 列表转换:全部大写
[for e in var.envs : upper(e)]

# 列表过滤:只要启用的
[for item in var.items : item.name if item.enabled]

# Map 转换:转换值
{for name, role in var.users : name => upper(role)}
```

## Day 12:Remote State & Backend

| Backend | Locking | 适用 |
|---------|---------|------|
| local | 无 | 个人学习、demo |
| S3 | DynamoDB | AWS 环境 |
| gcs | 原生支持 | GCP 环境 |
| pg | Advisory lock | 不想依赖云存储 |

## Day 13:Workspace — 多环境管理

```bash
terraform workspace new dev
terraform workspace new prod
terraform workspace select prod

# 在 .tf 中引用:
${terraform.workspace}  # "dev" 或 "prod"
```

Workspace = 同一套配置的多个 state 实例。

Workspace 适合:环境参数差异小(同一个 region,同一套 provider)。
目录分离适合:环境结构差异大(比如 dev 用 Docker,prod 用 K8s)

## Day 14:第2周综合练习

In [ ]:
print("=" * 60)
print("第2周综合练习交付清单")
print("=" * 60)

print("""
项目结构:
terraform-docker-app/
├── main.tf                    # 调用 modules
├── envs/
│   ├── dev.tfvars
│   └── prod.tfvars
└── modules/
    ├── network/               # Docker 网络 module
    ├── web/                   # Web 应用 module
    └── database/              # PostgreSQL module

要求:
  - 每个 Module 都有 version 参数
  - 使用 for_each 管理多实例 (dev 1实例, prod 3实例)
  - 使用 workspace 区分 dev / prod
  - 使用条件表达式 (prod 大规格, dev 小规格)
  - terraform plan 分别验证 dev 和 prod 的变更计划
""")

print("=" * 60)
print("第2周核心收获:")
print("1. Module = 可复用的基础设施组件,参数化差异")
print("2. 优先用 for_each(key 确定,增删不影响其他)")
print("3. 条件 + 函数 = 灵活的资源行为控制")
print("4. Remote Backend + Locking → 多人协作安全")
print("5. Workspace = 同一配置多环境,目录分离 = 差异大多套配置")
print("=" * 60)